# Evolving a quantum compiler pass — your own GPU

The same search as `01_gateway_quickstart.ipynb`, but the model runs on **this
instance's GPU** via vLLM or SGLang. Inference is then free per token, so the run
can be much longer — 100 generations rather than 30.

> **Requires a GPU instance.** Launch one from the **On-Demand** tab of the
> [qBraid dashboard](https://account.qbraid.com/dashboard). The GPU bills per
> minute while it runs, so terminate it when you are done (last cell).

In [ ]:
import os
import pathlib
import sys

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("working from:", ROOT)

## 1. What GPU do we have?

This decides how big a model you can serve. See `docs/CHOOSING_A_MODEL.md`.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv

## 2. Start the server

vLLM and SGLang both speak the OpenAI API and either works. vLLM is usually
easier to install; SGLang tends to be faster under concurrency.

Model sizing, roughly:

| VRAM | comfortable at bf16 |
|---|---|
| 24 GB | 7B |
| 48 GB | 14B |
| 80 GB | 32B |

**Size up if you can.** ShinkaEvolve asks the model to emit exactly-matching
SEARCH/REPLACE diffs, and models below ~14B fail at that often enough to stall a
run without ever erroring. `docs/CHOOSING_A_MODEL.md` covers this properly.

In [ ]:
!pip install -q vllm

The server must keep running while you evolve, so start it in the **background**
and wait for it to come up. The first run also downloads the weights, which on a
cold cache can take many minutes.

Alternatively, open a terminal and run `bash setup/serve_vllm.sh` there.

In [ ]:
import subprocess

MODEL = "Qwen/Qwen2.5-Coder-14B-Instruct"
env = dict(os.environ, MODEL=MODEL, PORT="8000")
server = subprocess.Popen(
    ["bash", "setup/serve_vllm.sh"],
    env=env,
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)
print("server pid", server.pid, "-- logs at /tmp/vllm.log")

In [ ]:
import time
import urllib.request

BASE = "http://localhost:8000/v1"
for attempt in range(180):
    try:
        urllib.request.urlopen(f"{BASE}/models", timeout=5)
        print("\nserver is up")
        break
    except Exception:
        if attempt % 10 == 0:
            print(f"waiting... {attempt * 10}s", end="\r")
        time.sleep(10)
else:
    print("\nserver did not come up -- check /tmp/vllm.log")

In [ ]:
!tail -5 /tmp/vllm.log

## 3. Pre-flight

No compatibility shim is needed here: the gateway's parameter restrictions do
not apply to a server you run yourself.

In [ ]:
!python setup/check_endpoint.py \
    --base-url http://localhost:8000/v1 \
    --model Qwen/Qwen2.5-Coder-14B-Instruct

## 4. Seed score

~3 seconds, no LLM. Establishes the starting point the search has to beat.

In [ ]:
!python task/evaluate.py --program_path task/initial.py --results_dir /tmp/seed_eval

## 5. Evolve

100 generations rather than 30 — inference costs nothing here, and the GPU bills
the same whether it is busy or idle.

Note that `--budget` is meaningless on a self-hosted endpoint: Shinka prices
every call at $0, so the cap can never trip. Bound the run with `--generations`
or wall clock instead.

In [ ]:
!python run_evolution.py --endpoint local \
    --base-url http://localhost:8000/v1 \
    --model Qwen/Qwen2.5-Coder-14B-Instruct \
    --generations 100 \
    --results-dir results/local_nb

## 6. Results

Check the **scored fraction** first. If most candidates failed, the model is
probably too small to follow the diff protocol — see `docs/CHOOSING_A_MODEL.md`.

In [ ]:
import sqlite3

import pandas as pd

rows = sqlite3.connect("results/local_nb/programs.sqlite").execute(
    "SELECT generation, combined_score, correct FROM programs ORDER BY generation"
).fetchall()
df = pd.DataFrame(rows, columns=["generation", "score", "correct"])
rate = df.correct.mean()
print(f"{int(df.correct.sum())}/{len(df)} candidates scored ({rate:.0%})")
if rate < 0.4:
    print("LOW -- the model is likely too small for the diff protocol.")
print(f"seed {df.score.iloc[0]:.4f}  ->  best {df.score.max():.4f}")

In [ ]:
import matplotlib.pyplot as plt

ok = df[df.correct == 1]
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(ok.generation, ok.score, s=24, label="candidate")
ax.plot(ok.generation, ok.score.cummax(), lw=2, color="crimson", label="best so far")
ax.axhline(1.0, ls="--", c="gray", lw=1, label="identity layout")
ax.set_xlabel("generation")
ax.set_ylabel("mean speedup vs identity layout")
ax.set_title("Evolution of the layout heuristic (self-hosted)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Verify the winner independently

In [ ]:
!python task/evaluate.py \
    --program_path results/local_nb/best/main.py \
    --results_dir /tmp/verify

## 8. Stop the server, then terminate the instance

The GPU bills per minute. Stopping the server is not enough — a *stopped*
instance still bills for storage, and **terminated** is the only zero-cost state.

Copy anything you want to keep off the instance first: on-demand filesystems are
deleted on terminate.

```bash
qbraid compute instances terminate <label> --yes
```

In [ ]:
server.terminate()
server.wait(timeout=30)
print("server stopped")